In [1]:
import pandas as pd
from rapidfuzz.fuzz import partial_ratio
import pykakasi
import re

In [2]:
kks = pykakasi.kakasi()
def romanize_japanese(text):
    result = kks.convert(text)
    return " ".join([item['hepburn'] for item in result])

In [3]:
def penalized_partial_ratio(a: str, b: str, penalty_strength: float = 0.1) -> float:
    a, b = a.lower(), b.lower()
    base_score = partial_ratio(a, b)
    length_difference = abs(len(a) - len(b))
    max_length = max(len(a), len(b))
    if not max_length:
        return base_score
    length_penalty = 1 - penalty_strength * (length_difference / max_length)
    return base_score * length_penalty

In [4]:
def clean_artist(name):
    if (not name) or not isinstance(name, str):
        return name
    name = name.lower()
    noise_words = ["official", "topic", "channel", "チャンネル", "chan'neru", "channeru", "chaneru", " ch.", " x ", "feat.", "vevo", "tv", "youtube", "virtual", "singer", "youtuber", "vtuber"]
    for word in noise_words:
        name = name.replace(word, " ").strip()
    return " ".join(name.split())

In [5]:
def sim_artist(artist, other):
    if not artist or not other:
        return 0
    artist = clean_artist(artist.lower())
    artist = romanize_japanese(str(artist)).lower()
    artist = clean_artist(artist)
    other = romanize_japanese(str(other)).lower()
    artists = [artist]
    if " " in artist:
        artists = [
            *artists,
            " ".join(reversed(artist.split(" ")))
        ]
    return max([penalized_partial_ratio(a, other) for a in artists])

In [6]:
def check_similarity(a, b):
    if not a or not b:
        return 0
    a = romanize_japanese(str(a)).lower()
    b = romanize_japanese(str(b)).lower()
    return penalized_partial_ratio(a, b)

In [7]:
df = pd.read_csv("matches.csv")

In [8]:
df = df.where(pd.notna(df), None)
df = df[df['yt_id'].notna() & (df['yt_id'] != '')]
df = df.sort_values("check", ascending=True).drop_duplicates(subset="filename", keep="last")
df = df.sort_values(["artist", "yt_channel", "title", "yt_title", "filename"], axis=0)
df['duplicated'] = df.duplicated('yt_id')
df['sim_artist'] = df.apply(lambda row: sim_artist(row['artist'], row['yt_channel']), axis=1)
df['sim_artist_title'] = df.apply(lambda row: sim_artist(row['artist'], row['yt_title']), axis=1)
df['sim_title'] = df.apply(lambda row: check_similarity(row['title'], row['yt_title']), axis=1)
df['sim_filename'] = df.apply(lambda row: check_similarity(row['filename'], row['yt_title']), axis=1)
df['pass'] = (df['sim_title'] >= 50) & ((df['sim_artist'] >= 50) | (df['sim_artist_title'] >= 50))
df['check'] = (df['check'] == True) | (df['check'] == 1) | (df['check'] == "TRUE") | (df['check'] == "True")
print(len(df))
df.tail(1)

7876


,filename,artist,title,acoustid_id,yt_query,yt_id,yt_channel,yt_title,duration,audio_format,...,score,yt_channel_id,yt_views,duplicated,sim_artist,sim_artist_title,sim_title,sim_filename,pass,check
6422,ㅤㅤ.mp3,None,None,NaN,ㅤㅤ,apFSQhcosac,𓆑,ㅤㅤ先生のこと好きになっちゃう ver.sekai,956.0,mhtml,...,105,@Yoeu-j2o,256468.0,False,0.0,0.0,0.0,52.358804,False,True


In [9]:
# df.loc[df_bug_index].tail(1)

In [10]:
# print(len(df))
# df = df[(df['audio_format'] != 'mhtml') & (df['yt_views'] > 0)]
# print(len(df))

In [11]:
# print(len(df))
# df = df[df['check'] | df['pass']]
# print(len(df))
# print(len(df[(df['check'] == True) & (df['pass'] != True)]))

In [12]:
# missing_fields = ["yt_query", "acoustid_id"]
# missing_fields = [f for f in missing_fields if f not in df.columns]
# df2 = pd.read_csv("matches - Copy (3).csv")
# if missing_fields:
#     df = df.merge(df2[["filename", *missing_fields]], on="filename", how="left")
# df.tail(1)

In [13]:
df['check'] = df['check'].astype('object')
df.loc[df['check'] != True, "check"] = None

In [14]:
df.loc[:, [
    "filename",
    "artist", "title",
    "acoustid_id",
    "yt_query", "yt_id",
    "yt_channel", "yt_title", "duration",
    "audio_format", "audio_codec", "audio_bitrate",
    "score",
    "yt_channel_id", "yt_views",
    "duplicated",
    "sim_artist",
    "sim_artist_title",
    "sim_title",
    "sim_filename",
    "pass",
    "check",
]].to_csv("matches.csv", index=False)

In [15]:
df.loc[:, [
    "filename", "yt_id",
    "artist", "yt_channel",
    "title", "yt_title", "yt_query", 
    "audio_format", "audio_codec",
    "audio_bitrate", "duration", "yt_views",
    "yt_channel_id", "duplicated",
    "sim_artist", "sim_artist_title", "sim_title", "sim_filename",
    "score", "pass", "check"
]].to_excel("matches.xlsx", index=False)